# 📦 05 – Final Load Preparation for Tableau
### Newton School of Technology · DVA Capstone 2
---
> **Purpose:** Engineer all features used in `04_statistical_analysis`, build every summary table that feeds the 18 charts, and export 7 Tableau-ready CSV files.
> **Source notebook:** `04_statistical_analysis_updated.ipynb`
> **Color Theme:** Warm Earth + Dark Navy — matches notebook 04 exactly
> **Output Files:**
> - `final_product_data.csv` — full product-level table (Page 2 primary source)
> - `category_summary.csv` — per-category KPIs (Charts 1, 2, 12, 13, 14, 16, 18)
> - `brand_summary.csv` — per-brand KPIs (Chart 15)
> - `gender_summary.csv` — per-gender KPIs (Chart 3, 11)
> - `inventory_summary.csv` — category × inventory level (Chart 13, 14)
> - `category_gender_summary.csv` — category × gender breakdown (Chart 17)
> - `price_gap_summary.csv` — original vs discount price gap (Chart 18)

---
## 🔹 Step 1 — Setup, Color Palette & Load Dataset

Replicate the exact same color palette and `plt.rcParams` from `04_statistical_analysis` so any
validation plots rendered here are visually consistent.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════
#  COLOR PALETTE — Warm Earth + Dark Navy  (identical to 04_statistical_analysis)
# ══════════════════════════════════════════════════════════════════════════
BG     = '#FAF6F1'   # warm off-white background
NAVY   = '#0D1F3C'   # dark navy  (primary text / strong bars)
BEIGE  = '#D9C4A8'   # light beige (neutral fills)
BROWN  = '#8B5E3C'   # light brown (secondary accent)
ORANGE = '#E07B39'   # warm orange (highlights / severe erosion)
TAN    = '#C4956A'   # caramel mid-tone (high erosion)
CREAM  = '#EDE0CC'   # cream light fill
SLATE  = '#3D5475'   # slate navy (axes / gridlines)

# Aliases — kept so any copied chart code runs without changes
VIOLET = BROWN
GOLD   = ORANGE
EMERALD = '#5A7A5A'
CORAL  = ORANGE
SKY    = SLATE
ROSE   = BROWN
AMBER  = TAN

PALETTE = [NAVY, BROWN, ORANGE, BEIGE, SLATE, TAN, CREAM, '#4A3728']

plt.rcParams.update({
    'figure.facecolor'  : BG,
    'axes.facecolor'    : '#FFFDF9',
    'axes.edgecolor'    : '#C8B89A',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.color'        : '#E8DDD0',
    'grid.linewidth'    : 0.7,
    'font.family'       : 'DejaVu Sans',
    'text.color'        : NAVY,
    'xtick.color'       : SLATE,
    'ytick.color'       : SLATE,
})

# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('CleanedData.csv')

print("✅ Dataset Loaded")
print(f"   Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Columns: {list(df.columns)}")
print()
print(df[['DiscountPrice','OriginalPrice','DiscountPercentage','SizeCount']].describe().round(2))

✅ Dataset Loaded
   Shape  : 526,564 rows × 12 columns
   Columns: ['URL', 'Product_id', 'BrandName', 'Category', 'Individual_category', 'category_by_Gender', 'Description', 'DiscountPrice', 'OriginalPrice', 'DiscountPercentage', 'SizeOption', 'SizeCount']

       DiscountPrice  OriginalPrice  DiscountPercentage  SizeCount
count      526564.00      526564.00           526564.00  526564.00
mean         1308.59        2414.07               41.98       4.58
std          1209.45        1916.96               24.18       2.37
min            99.00          99.00                0.00       1.00
25%           659.45        1299.00               24.00       4.00
50%           967.50        1999.00               50.00       5.00
75%          1536.50        2899.00               60.00       6.00
max         90000.00       90000.00               90.00      80.00


---
## 🔹 Step 2 — Final Cleaning

Remove nulls, fix data types, standardise text — identical rules to `04_statistical_analysis` Section 0.

In [2]:
# ── Null check before ─────────────────────────────────────────────────────────
print("Null values BEFORE cleaning:")
print(df.isnull().sum())

# ── Drop nulls in critical columns ────────────────────────────────────────────
critical_cols = ['DiscountPrice', 'OriginalPrice', 'DiscountPercentage',
                 'Category', 'BrandName', 'SizeCount', 'category_by_Gender']
df = df.dropna(subset=critical_cols)

# ── Fix datatypes ──────────────────────────────────────────────────────────────
df['DiscountPrice']      = pd.to_numeric(df['DiscountPrice'],      errors='coerce')
df['OriginalPrice']      = pd.to_numeric(df['OriginalPrice'],      errors='coerce')
df['DiscountPercentage'] = pd.to_numeric(df['DiscountPercentage'], errors='coerce')
df['SizeCount']          = pd.to_numeric(df['SizeCount'],          errors='coerce').astype('Int64')
df = df.dropna(subset=['DiscountPrice','OriginalPrice','DiscountPercentage','SizeCount'])

# ── Standardise text ───────────────────────────────────────────────────────────
df['Category']           = df['Category'].str.strip().str.title()
df['BrandName']          = df['BrandName'].str.strip().str.title()
df['category_by_Gender'] = df['category_by_Gender'].str.strip().str.title()

# ── Remove impossible values ───────────────────────────────────────────────────
df = df[df['OriginalPrice'] > 0]
df = df[df['DiscountPrice'] > 0]
df = df[(df['DiscountPercentage'] >= 0) & (df['DiscountPercentage'] <= 90)]
df = df.reset_index(drop=True)

print(f"\n✅ After cleaning: {df.shape[0]:,} rows remain")
print("Null values AFTER cleaning:")
print(df.isnull().sum())

Null values BEFORE cleaning:
URL                    0
Product_id             0
BrandName              0
Category               0
Individual_category    0
category_by_Gender     0
Description            0
DiscountPrice          0
OriginalPrice          0
DiscountPercentage     0
SizeOption             0
SizeCount              0
dtype: int64

✅ After cleaning: 526,564 rows remain
Null values AFTER cleaning:
URL                    0
Product_id             0
BrandName              0
Category               0
Individual_category    0
category_by_Gender     0
Description            0
DiscountPrice          0
OriginalPrice          0
DiscountPercentage     0
SizeOption             0
SizeCount              0
dtype: int64


---
## 🔹 Step 3 — Feature Engineering

Seven engineered columns — each derived from the exact logic used inside the charts of `04_statistical_analysis`.

| Column | Derived from | Used in |
|---|---|---|
| `Price_Segment` | `OriginalPrice` thresholds | Charts 2, 10 |
| `Discount_Segment` | `DiscountPercentage` thresholds | Chart 12 |
| `Inventory_Level` | `SizeCount` thresholds | Charts 13, 14 |
| `Value_for_Money` | `DiscountPercentage / OriginalPrice` | product_level |
| `Value_Lost` | `OriginalPrice − DiscountPrice` | Charts 16, 18 |
| `Profit_Erosion_Flag` | `DiscountPercentage` vs 50/42 thresholds | Chart 16 `rate_color()` |
| `Discount_Health` | Brand avg vs platform avg discount | Chart 15 `bar_cols` |

In [3]:
# ── 1. Price Segment  ─────────────────────────────────────────────────────────
#    Matches price_segment() in Section 1 of notebook 04
#    Powers: Chart 2 stacked bar, seg_order filter
def price_segment(p):
    if p < 500:      return 'Low  (<₹500)'
    elif p <= 1500:  return 'Mid  (₹500–1500)'
    else:            return 'Premium  (>₹1500)'

df['Price_Segment'] = df['OriginalPrice'].apply(price_segment)
seg_order = ['Low  (<₹500)', 'Mid  (₹500–1500)', 'Premium  (>₹1500)']

# ── 2. Discount Segment  ──────────────────────────────────────────────────────
#    Matches discount_segment() in Section 5 of notebook 04
#    Powers: Chart 12 stacked bar, disc_order filter
def discount_segment(d):
    if d < 30:      return 'Low (<30%)'
    elif d <= 60:   return 'Medium (30–60%)'
    else:           return 'High (>60%)'

df['Discount_Segment'] = df['DiscountPercentage'].apply(discount_segment)
disc_order = ['Low (<30%)', 'Medium (30–60%)', 'High (>60%)']

# ── 3. Inventory Level  ───────────────────────────────────────────────────────
#    Powers: Chart 13 bar, Chart 14 bubble, inventory_summary table
def inventory_level(s):
    if s <= 3:   return 'Low (1–3 sizes)'
    elif s <= 6: return 'Medium (4–6 sizes)'
    else:        return 'High (7+ sizes)'

df['Inventory_Level'] = df['SizeCount'].apply(inventory_level)

# ── 4. Value for Money  ───────────────────────────────────────────────────────
#    Discount % per ₹1 of original price — higher = better deal
df['Value_for_Money'] = (df['DiscountPercentage'] / df['OriginalPrice']).round(6)

# ── 5. Absolute Value Lost  ───────────────────────────────────────────────────
#    Feeds all erosion visuals — Charts 16, 18, category_summary
df['Value_Lost'] = (df['OriginalPrice'] - df['DiscountPrice']).round(2)

# ── 6. Profit Erosion Flag  ───────────────────────────────────────────────────
#    Mirrors rate_color() in Chart 16:
#      ORANGE = severe (≥50%), TAN = high (42–50%), BEIGE = manageable
def erosion_flag(d):
    if d >= 50:   return 'Severe (>50%)'
    elif d >= 42: return 'High (42–50%)'
    else:         return 'Manageable (<42%)'

df['Profit_Erosion_Flag'] = df['DiscountPercentage'].apply(erosion_flag)

# ── 7. Discount Health  ───────────────────────────────────────────────────────
#    Mirrors bar_cols logic in Chart 15:
#      ORANGE bar if d > mean_disc (top-15 avg), else BEIGE
#    Here we use the platform-wide average as the threshold
platform_avg_discount = df['DiscountPercentage'].mean()
df['Discount_Health'] = df['DiscountPercentage'].apply(
    lambda d: 'High Discounter' if d > platform_avg_discount else 'Price Disciplined'
)

print("✅ Feature Engineering Complete")
print()
print("New columns added:")
for col in ['Price_Segment','Discount_Segment','Inventory_Level',
            'Value_for_Money','Value_Lost','Profit_Erosion_Flag','Discount_Health']:
    print(f"   {col}: {df[col].unique()[:3].tolist()}")
print(f"\n   Platform avg discount (Discount_Health threshold): {platform_avg_discount:.1f}%")
print(f"   Price_Segment dist:\n{df['Price_Segment'].value_counts()[seg_order]}")
print(f"   Discount_Segment dist:\n{df['Discount_Segment'].value_counts()[disc_order]}")

✅ Feature Engineering Complete

New columns added:
   Price_Segment: ['Mid  (₹500–1500)', 'Premium  (>₹1500)', 'Low  (<₹500)']
   Discount_Segment: ['Medium (30–60%)', 'Low (<30%)', 'High (>60%)']
   Inventory_Level: ['Medium (4–6 sizes)', 'High (7+ sizes)', 'Low (1–3 sizes)']
   Value_for_Money: [0.03002, 0.047868, 0.039314]
   Value_Lost: [674.55, 631.95, 769.45]
   Profit_Erosion_Flag: ['High (42–50%)', 'Severe (>50%)', 'Manageable (<42%)']
   Discount_Health: ['High Discounter', 'Price Disciplined']

   Platform avg discount (Discount_Health threshold): 42.0%
   Price_Segment dist:
Price_Segment
Low  (<₹500)          12784
Mid  (₹500–1500)     166610
Premium  (>₹1500)    347170
Name: count, dtype: int64
   Discount_Segment dist:
Discount_Segment
Low (<30%)         147502
Medium (30–60%)    255909
High (>60%)        123153
Name: count, dtype: int64


---
## 🔹 Step 4 — Build Summary Tables

One table per chart group — every aggregation mirrors what the chart code computes inside notebook 04.

### 🟢 Page 1 — Executive Overview (Charts 1, 2, 3, 17)

In [4]:
# ── category_summary  ────────────────────────────────────────────────────────
#    Feeds: Chart 1 (avg price lollipop), Chart 2 (price segment stacked bar),
#           Chart 12 (discount segment), Chart 13 (size count), Chart 16 (erosion),
#           Chart 18 (column price gap)
category_summary = df.groupby('Category').agg(
    Product_Count       = ('Product_id',          'count'),
    Avg_Original_Price  = ('OriginalPrice',        'mean'),
    Avg_Discount_Price  = ('DiscountPrice',        'mean'),
    Avg_Discount_Pct    = ('DiscountPercentage',   'mean'),
    Total_Original_Value= ('OriginalPrice',        'sum'),
    Total_Revenue       = ('DiscountPrice',        'sum'),
    Total_Value_Lost    = ('Value_Lost',           'sum'),
    Avg_Value_Lost      = ('Value_Lost',           'mean'),   # Chart 18 gap label
    Avg_Size_Count      = ('SizeCount',            'mean')    # Chart 13 bar height
).reset_index().round(2)

# Erosion Rate — matches erosion['Erosion_Rate'] computed in Section 7 of notebook 04
category_summary['Erosion_Rate_Pct'] = (
    category_summary['Total_Value_Lost'] / category_summary['Total_Original_Value'] * 100
).round(2)

# Erosion Flag — matches rate_color() thresholds in Chart 16
def cat_erosion_flag(r):
    if r >= 50:   return 'Severe (>50%)'
    elif r >= 42: return 'High (42–50%)'
    else:         return 'Manageable (<42%)'

category_summary['Erosion_Flag'] = category_summary['Erosion_Rate_Pct'].apply(cat_erosion_flag)
category_summary = category_summary.sort_values('Avg_Original_Price', ascending=False).reset_index(drop=True)

print("category_summary:")
print(category_summary.to_string(index=False))

category_summary:
                Category  Product_Count  Avg_Original_Price  Avg_Discount_Price  Avg_Discount_Pct  Total_Original_Value  Total_Revenue  Total_Value_Lost  Avg_Value_Lost  Avg_Size_Count  Erosion_Rate_Pct      Erosion_Flag
             Indian Wear         145845             3593.23             1606.61             52.81           524055138.8   234316286.15      289738852.61         1986.62            3.21             55.29     Severe (>50%)
             Bottom Wear          55439             2407.30             1550.37             35.92           133458032.0    85951153.20       47506878.80          856.92            5.51             35.60 Manageable (<42%)
               Plus Size          13496             2200.86              979.68             55.14            29702803.0    13221789.63       16481013.37         1221.18            5.96             55.49     Severe (>50%)
             Sports Wear          20627             2049.73             1236.94             38.17 

In [5]:
# ── gender_summary  ──────────────────────────────────────────────────────────
#    Feeds: Chart 3 (gender donut), Chart 11 data reference
gender_summary = df.groupby('category_by_Gender').agg(
    Product_Count      = ('Product_id',        'count'),
    Avg_Original_Price = ('OriginalPrice',      'mean'),
    Avg_Discount_Price = ('DiscountPrice',      'mean'),
    Avg_Discount_Pct   = ('DiscountPercentage', 'mean'),
    Avg_Value_Lost     = ('Value_Lost',         'mean'),
    Total_Value_Lost   = ('Value_Lost',         'sum')
).reset_index().round(2)

gender_summary['Pct_of_Catalogue'] = (
    gender_summary['Product_Count'] / gender_summary['Product_Count'].sum() * 100
).round(2)

print("gender_summary:")
print(gender_summary.to_string(index=False))

gender_summary:
category_by_Gender  Product_Count  Avg_Original_Price  Avg_Discount_Price  Avg_Discount_Pct  Avg_Value_Lost  Total_Value_Lost  Pct_of_Catalogue
               Men         187379             2318.09             1368.34              37.6          949.75      177963670.18             35.59
             Women         339185             2467.09             1275.58              44.4         1191.51      404143019.01             64.41


In [6]:
# ── category_gender_summary  ─────────────────────────────────────────────────
#    Feeds: Chart 17 detailed pie — matches cat_gender groupby in Chart 17 code
category_gender_summary = (
    df.groupby(['Category', 'category_by_Gender'])
      .agg(
          Product_Count      = ('Product_id',        'count'),
          Avg_Original_Price = ('OriginalPrice',      'mean'),
          Avg_Discount_Price = ('DiscountPrice',      'mean'),
          Avg_Discount_Pct   = ('DiscountPercentage', 'mean'),
          Avg_Value_Lost     = ('Value_Lost',         'mean')
      )
      .reset_index()
      .round(2)
)
# Pie_Label matches cat_gender['Label'] in Chart 17
category_gender_summary['Pie_Label'] = (
    category_gender_summary['Category'] + ' (' +
    category_gender_summary['category_by_Gender'] + ')'
)
category_gender_summary = category_gender_summary.sort_values('Product_Count', ascending=False).reset_index(drop=True)

print("category_gender_summary (Chart 17 feed):")
print(category_gender_summary.to_string(index=False))

category_gender_summary (Chart 17 feed):
                Category category_by_Gender  Product_Count  Avg_Original_Price  Avg_Discount_Price  Avg_Discount_Pct  Avg_Value_Lost                      Pie_Label
                 Western              Women         140992             2048.93             1224.11             40.31          824.81                Western (Women)
             Indian Wear              Women         119911             3524.80             1578.84             52.81         1945.96            Indian Wear (Women)
                 Topwear                Men          74537             2041.36             1279.80             36.70          761.56                  Topwear (Men)
             Bottom Wear                Men          55439             2407.30             1550.37             35.92          856.92              Bottom Wear (Men)
   Lingerie & Sleep Wear              Women          55258             1425.51              856.86             35.14          568.65  Linge

### 🔵 Page 2 — Pricing & Discounts (Charts 4–10, 12, 18)

In [7]:
# ── product_level  ───────────────────────────────────────────────────────────
#    Primary Tableau source for Page 2 — all product-level charts
#    Contains every engineered column used in charts 4–10, 12
product_level = df[[
    'Product_id', 'BrandName', 'Category', 'Individual_category',
    'category_by_Gender', 'OriginalPrice', 'DiscountPrice',
    'DiscountPercentage', 'SizeCount',
    'Price_Segment',         # Chart 2 filter dimension
    'Discount_Segment',      # Chart 12 filter dimension
    'Inventory_Level',       # Chart 13/14 filter dimension
    'Value_for_Money',       # customer value measure
    'Value_Lost',            # erosion measure
    'Profit_Erosion_Flag',   # Chart 16 colour dimension
    'Discount_Health'        # Chart 15 colour dimension
]].copy()

print("product_level preview:")
print(product_level.head(3).to_string())
print(f"\nShape: {product_level.shape}")

product_level preview:
   Product_id   BrandName     Category Individual_category category_by_Gender  OriginalPrice  DiscountPrice  DiscountPercentage  SizeCount     Price_Segment Discount_Segment     Inventory_Level  Value_for_Money  Value_Lost Profit_Erosion_Flag  Discount_Health
0     2296012    Roadster  Bottom Wear               Jeans                Men         1499.0         824.45                45.0          5  Mid  (₹500–1500)  Medium (30–60%)  Medium (4–6 sizes)         0.030020      674.55       High (42–50%)  High Discounter
1    13780156  Locomotive  Bottom Wear         Track-pants                Men         1149.0         517.05                55.0          4  Mid  (₹500–1500)  Medium (30–60%)  Medium (4–6 sizes)         0.047868      631.95       Severe (>50%)  High Discounter
2    11895958    Roadster      Topwear              Shirts                Men         1399.0         629.55                55.0          6  Mid  (₹500–1500)  Medium (30–60%)  Medium (4–6 sizes)    

In [8]:
# ── price_gap_summary  ───────────────────────────────────────────────────────
#    Feeds: Chart 18 column chart — mirrors cat_price groupby in Chart 18 code
price_gap_summary = (
    df.groupby('Category')
      .agg(
          Avg_Original_Price = ('OriginalPrice',  'mean'),
          Avg_Discount_Price = ('DiscountPrice',  'mean'),
          Product_Count      = ('Product_id',     'count')
      )
      .reset_index()
      .round(2)
)
# Gap and Erosion_Pct — mirrors the Chart 18 gap label calculation
price_gap_summary['Avg_Value_Eroded'] = (
    price_gap_summary['Avg_Original_Price'] - price_gap_summary['Avg_Discount_Price']
).round(2)
price_gap_summary['Erosion_Pct'] = (
    price_gap_summary['Avg_Value_Eroded'] / price_gap_summary['Avg_Original_Price'] * 100
).round(2)
price_gap_summary['Erosion_Flag'] = price_gap_summary['Erosion_Pct'].apply(
    lambda r: 'Severe (>50%)' if r >= 50 else ('High (42–50%)' if r >= 42 else 'Manageable (<42%)')
)
price_gap_summary = price_gap_summary.sort_values('Avg_Original_Price', ascending=False).reset_index(drop=True)

print("price_gap_summary (Chart 18 feed):")
print(price_gap_summary.to_string(index=False))

price_gap_summary (Chart 18 feed):
                Category  Avg_Original_Price  Avg_Discount_Price  Product_Count  Avg_Value_Eroded  Erosion_Pct      Erosion_Flag
             Indian Wear             3593.23             1606.61         145845           1986.62        55.29     Severe (>50%)
             Bottom Wear             2407.30             1550.37          55439            856.93        35.60 Manageable (<42%)
               Plus Size             2200.86              979.68          13496           1221.18        55.49     Severe (>50%)
             Sports Wear             2049.73             1236.94          20627            812.79        39.65 Manageable (<42%)
                 Western             2048.93             1224.11         140992            824.82        40.26 Manageable (<42%)
                 Topwear             2041.36             1279.80          74537            761.56        37.31 Manageable (<42%)
   Lingerie & Sleep Wear             1425.51              856.

### 🟣 Page 3 — Brand & Inventory Intelligence (Charts 13–16)

In [9]:
# ── brand_summary  ───────────────────────────────────────────────────────────
#    Feeds: Chart 15 brand lollipop — mirrors top15 groupby in Chart 15 code
brand_summary = df.groupby('BrandName').agg(
    Total_Products     = ('Product_id',        'count'),
    Avg_Original_Price = ('OriginalPrice',      'mean'),
    Avg_Discount_Price = ('DiscountPrice',      'mean'),
    Avg_Discount_Pct   = ('DiscountPercentage', 'mean'),
    Total_Value_Lost   = ('Value_Lost',         'sum'),
    Avg_Size_Count     = ('SizeCount',          'mean')
).reset_index().round(2)

# Discount_Health — mirrors bar_cols in Chart 15:
#   orange if Avg_Discount_Pct > mean_disc of top-15, else beige
#   Here we use brand_summary's own mean as threshold (consistent with chart logic)
brand_mean_disc = brand_summary['Avg_Discount_Pct'].mean()
brand_summary['Discount_Health'] = brand_summary['Avg_Discount_Pct'].apply(
    lambda d: 'High Discounter' if d > brand_mean_disc else 'Price Disciplined'
)

brand_summary = brand_summary.sort_values('Total_Products', ascending=False).reset_index(drop=True)

print(f"Top 15 brands (Chart 15 source — threshold avg: {brand_mean_disc:.1f}%):")
print(brand_summary.head(15).to_string(index=False))

Top 15 brands (Chart 15 source — threshold avg: 38.7%):
            BrandName  Total_Products  Avg_Original_Price  Avg_Discount_Price  Avg_Discount_Pct  Total_Value_Lost  Avg_Size_Count   Discount_Health
               Pothys           16005             2704.33             1808.68             30.11       14334875.60             1.0 Price Disciplined
             Roadster           10935             1600.58              763.01             50.54        9158904.22             5.2   High Discounter
               Kalini            9589             2596.23              690.69             72.36       18272231.09            2.18   High Discounter
             Here&Now            6515             2032.00              753.98             62.27        8326279.22            5.57   High Discounter
Hrx By Hrithik Roshan            5297             2010.79             1005.42             49.49        5325418.75             5.1   High Discounter
       Mast & Harbour            5148             1664.0

In [10]:
# ── inventory_summary  ───────────────────────────────────────────────────────
#    Feeds: Chart 14 bubble (AvgSize), Chart 13 bar (Avg_Size_Count)
inventory_summary = df.groupby(['Category', 'Inventory_Level']).agg(
    Product_Count  = ('Product_id',        'count'),
    Avg_Size_Count = ('SizeCount',         'mean'),
    Avg_Discount   = ('DiscountPercentage','mean'),
    Avg_Value_Lost = ('Value_Lost',        'mean')
).reset_index().round(2)

print("inventory_summary:")
print(inventory_summary.to_string(index=False))

inventory_summary:
                Category    Inventory_Level  Product_Count  Avg_Size_Count  Avg_Discount  Avg_Value_Lost
             Bottom Wear    High (7+ sizes)          10118            7.98         29.76          822.50
             Bottom Wear    Low (1–3 sizes)           2554            1.95         21.33          538.26
             Bottom Wear Medium (4–6 sizes)          42767            5.14         38.24          884.09
             Indian Wear    High (7+ sizes)          11086            7.39         59.74         1894.43
             Indian Wear    Low (1–3 sizes)          74248            1.04         51.00         2065.42
             Indian Wear Medium (4–6 sizes)          60511            5.11         53.77         1906.82
Inner Wear &  Sleep Wear    High (7+ sizes)            464             7.4         14.58          169.22
Inner Wear &  Sleep Wear    Low (1–3 sizes)           7157            1.71         21.66          261.71
Inner Wear &  Sleep Wear Medium (4–6

---
## 🔹 Step 5 — Export All CSVs

Export 7 Tableau-ready files — one for each chart group.

In [11]:
import os
output_path = '.'   # change if needed

files = {
    'final_product_data.csv'      : product_level,             # Page 2 — all product-level charts
    'category_summary.csv'        : category_summary,          # Charts 1, 2, 12, 13, 16, 18
    'brand_summary.csv'           : brand_summary,             # Chart 15 brand lollipop
    'gender_summary.csv'          : gender_summary,            # Chart 3 gender donut
    'inventory_summary.csv'       : inventory_summary,         # Charts 13, 14
    'category_gender_summary.csv' : category_gender_summary,   # Chart 17 detailed pie
    'price_gap_summary.csv'       : price_gap_summary,         # Chart 18 column gap
}

for fname, dataframe in files.items():
    dataframe.to_csv(os.path.join(output_path, fname), index=False)
    print(f"✅ Saved: {fname:40s}  →  {dataframe.shape[0]:,} rows × {dataframe.shape[1]} cols")

print()
print("🎉 All 7 Tableau-ready files exported successfully!")

✅ Saved: final_product_data.csv                    →  526,564 rows × 16 cols
✅ Saved: category_summary.csv                      →  8 rows × 12 cols
✅ Saved: brand_summary.csv                         →  2,087 rows × 8 cols
✅ Saved: gender_summary.csv                        →  2 rows × 8 cols
✅ Saved: inventory_summary.csv                     →  24 rows × 6 cols
✅ Saved: category_gender_summary.csv               →  11 rows × 8 cols
✅ Saved: price_gap_summary.csv                     →  8 rows × 7 cols

🎉 All 7 Tableau-ready files exported successfully!


---
## 🔹 Step 6 — Validation

Reload every file and confirm zero nulls.

In [12]:
print("=== Validation Check ===\n")
for fname in files:
    temp  = pd.read_csv(fname)
    nulls = temp.isnull().sum().sum()
    status = '✅' if nulls == 0 else '⚠️'
    print(f"{status} {fname:40s}  |  {temp.shape[0]:>8,} rows  |  {temp.shape[1]:>2} cols  |  {nulls} nulls")

print()
print("All files clean and ready for Tableau. 🚀")

=== Validation Check ===

✅ final_product_data.csv                    |   526,564 rows  |  16 cols  |  0 nulls
✅ category_summary.csv                      |         8 rows  |  12 cols  |  0 nulls
✅ brand_summary.csv                         |     2,087 rows  |   8 cols  |  0 nulls
✅ gender_summary.csv                        |         2 rows  |   8 cols  |  0 nulls
✅ inventory_summary.csv                     |        24 rows  |   6 cols  |  0 nulls
✅ category_gender_summary.csv               |        11 rows  |   8 cols  |  0 nulls
✅ price_gap_summary.csv                     |         8 rows  |   7 cols  |  0 nulls

All files clean and ready for Tableau. 🚀


---
## 📋 Tableau Loading Guide

| File | Connect To | Powers Charts |
|---|---|---|
| `final_product_data.csv` | Data Source (Primary) | 4, 5, 6, 7, 8, 9, 10, 12 |
| `category_summary.csv` | Data Source | 1, 2, 13, 16, 18 |
| `brand_summary.csv` | Data Source | 15 |
| `gender_summary.csv` | Data Source | 3 |
| `inventory_summary.csv` | Data Source | 13, 14 |
| `category_gender_summary.csv` | Data Source | 17 |
| `price_gap_summary.csv` | Data Source | 18 |

### 🔑 Key Dimensions for Filters
| Column | File | Tableau Page |
|---|---|---|
| `Price_Segment` | final_product_data | Page 1 & 2 |
| `Discount_Segment` | final_product_data | Page 2 |
| `Inventory_Level` | final_product_data / inventory_summary | Page 3 |
| `category_by_Gender` | final_product_data / gender_summary | All pages |
| `Profit_Erosion_Flag` | final_product_data / category_summary | Page 3 (Chart 16 colour) |
| `Discount_Health` | final_product_data / brand_summary | Page 3 (Chart 15 colour) |

### 🔑 Key Measures
| Measure | Derivation | Used in |
|---|---|---|
| `Value_Lost` | `OriginalPrice − DiscountPrice` | Charts 16, 18 |
| `Erosion_Rate_Pct` | `Total_Value_Lost / Total_Original_Value × 100` | Chart 16 |
| `Avg_Value_Eroded` | `Avg_Original − Avg_Discount` per category | Chart 18 gap label |
| `Value_for_Money` | `DiscountPercentage / OriginalPrice` | Page 2 scatter |
| `Discount_Health` | vs platform avg discount | Chart 15 bar colour |

### 🔗 Exact Sync with Notebook 04 Logic
| Notebook 04 code | Notebook 05 column |
|---|---|
| `price_segment()` thresholds: <500, ≤1500 | `Price_Segment` |
| `discount_segment()` thresholds: <30, ≤60 | `Discount_Segment` |
| `inventory_level()` thresholds: ≤3, ≤6 | `Inventory_Level` |
| Chart 15 `bar_cols`: orange if `d > mean_disc` | `brand_summary.Discount_Health` |
| Chart 16 `rate_color()`: orange ≥50%, tan 42–50% | `Profit_Erosion_Flag`, `Erosion_Flag` |
| Chart 17 `cat_gender['Label']` | `category_gender_summary.Pie_Label` |
| Chart 18 gap = `Avg_Original − Avg_Discount` | `price_gap_summary.Avg_Value_Eroded` |